In [ ]:
import numpy as np
import pandas as pd

def predict_next_month_from_neighbors(q_idx, k=20, sim_min=0.80):
    """
    q_idx: monthly_chunks 的行号（query chunk）
    返回：mu_hat, p10, p50, p90, confidence, top1_sim
    """
    q_vec = X[q_idx].reshape(1, -1)
    dist, idx = nn.kneighbors(q_vec, n_neighbors=max(k+5, 30))
    idx = idx.flatten()
    dist = dist.flatten()
    sim = 1 - dist

    # 排除自己
    pairs = [(i, float(s)) for i, s in zip(idx, sim) if i != q_idx]
    pairs = pairs[:k]

    neigh = monthly_chunks.iloc[[i for i,_ in pairs]].copy()
    neigh["sim"] = [s for _,s in pairs]

    # 只用有 label 的 neighbors
    neigh = neigh.dropna(subset=["label_next_month_ret"])
    if len(neigh) < 5:
        return None  # 数据不足，不交易

    top1_sim = float(neigh["sim"].max())
    confidence = float(neigh["sim"].mean())

    # similarity 加权均值
    w = neigh["sim"].clip(lower=0)
    w = w / (w.sum() + 1e-12)
    mu_hat = float((w * neigh["label_next_month_ret"].astype(float)).sum())

    # 用 neighbors 的分布做风险
    p10 = float(neigh["label_next_month_ret"].quantile(0.10))
    p50 = float(neigh["label_next_month_ret"].quantile(0.50))
    p90 = float(neigh["label_next_month_ret"].quantile(0.90))

    # 门槛：相似度太低就不做
    if top1_sim < sim_min:
        return None

    return {
        "mu": mu_hat,
        "p10": p10,
        "p50": p50,
        "p90": p90,
        "confidence": confidence,
        "top1_sim": top1_sim,
        "n_neighbors_used": int(len(neigh))
    }

In [ ]:
from pathlib import Path

# 你可以调这些超参
K = 20
SIM_MIN = 0.85
WEIGHT_CAP = 0.6

# 用每个 symbol 每个月的 chunk 来生成当月仓位
# 这里以 month 为单位生成 weights_by_month，然后映射到月初交易日执行

# 先做 month -> rebalance_date（用 prices 的交易日）
reb_dates = (
    pd.Series(px.index, index=px.index)
    .groupby(px.index.to_period("M"))
    .min()
    .sort_values()
)

symbols = sorted(monthly_chunks["symbol"].unique())
months = sorted(monthly_chunks["month"].unique())

weights_list = []
pred_list = []

for mon in months:
    # 该月的 rebalance date（月初）
    period = pd.Period(mon)
    if period not in reb_dates.index:
        continue
    reb_date = reb_dates.loc[period]

    raw_w = {}
    meta = {}

    for sym in symbols:
        mask = (monthly_chunks["symbol"]==sym) & (monthly_chunks["month"]==mon)
        if mask.sum() == 0:
            continue
        q_idx = int(np.where(mask.to_numpy())[0][0])

        pred = predict_next_month_from_neighbors(q_idx, k=K, sim_min=SIM_MIN)
        if pred is None:
            raw = 0.0
        else:
            # 仓位强度：收益信号 * 相似度置信度 / 下行风险
            downside = abs(pred["p10"]) + 1e-6
            raw = max(pred["mu"] * pred["confidence"] / downside, 0.0)

            meta = {**pred, "symbol": sym, "month": mon, "rebalance_date": reb_date}
            pred_list.append(meta)

        raw_w[sym] = float(raw)

    # 归一化成组合权重（允许持有现金）
    s = sum(raw_w.values())
    if s <= 1e-12:
        continue

    # 归一化到总仓位=1（你也可以改成总仓位=0.7 等）
    w = {k: v/s for k,v in raw_w.items()}
    # cap 再归一化
    w = {k: min(v, WEIGHT_CAP) for k,v in w.items()}
    s2 = sum(w.values())
    w = {k: (v/s2 if s2>1e-12 else 0.0) for k,v in w.items()}

    for sym, ww in w.items():
        weights_list.append({"rebalance_date": reb_date, "symbol": sym, "weight": ww})

W_sig = pd.DataFrame(weights_list)
pred_df = pd.DataFrame(pred_list)

W_sig.head(), pred_df.head()

In [ ]:
# 转成宽表
W_wide = W_sig.pivot(index="rebalance_date", columns="symbol", values="weight").fillna(0.0).sort_index()

# 用之前的 backtest 函数
result_sim, W_exec_sim = backtest_weights_returns(W_wide, ret_df, fee_rate=0.0005, lag_weights=1)

metrics_sim = perf(result_sim["nav"], result_sim["net_ret"])
metrics_sim